In [ ]:
!pip install -U "bitsandbytes>=0.46.1" accelerate

In [ ]:
# Phase 1: Dataset Loader — Code-Mixed Hinglish Hate Speech Dataset

import pandas as pd
import os
DATA_PATH = "/content/combined_hate_speech_dataset.csv"

# Loading the dataset
df = pd.read_csv(DATA_PATH)

print("Original Columns:", df.columns.tolist())
print("Original Shape:", df.shape)

# Auto-detect columns
text_col = None
label_col = None
for c in df.columns:
    cl = c.lower()
    if text_col is None and any(x in cl for x in ["text", "comment", "tweet", "content"]):
        text_col = c
    if label_col is None and any(x in cl for x in ["label", "toxic", "hate", "class"]):
        label_col = c
if text_col is None or label_col is None:
    raise ValueError("Could not detect text or label column.")

print("Detected text column:", text_col)
print("Detected label column:", label_col)

# Normalize dataset
clean_df = df[[text_col, label_col]].rename(
    columns={text_col: "text", label_col: "is_toxic"}
)

# Convert types
clean_df["text"] = clean_df["text"].astype(str)
clean_df["is_toxic"] = pd.to_numeric(clean_df["is_toxic"], errors="coerce")

# Remove invalid rows
clean_df = clean_df.dropna()
clean_df = clean_df[clean_df["text"].str.strip() != ""]

# Convert labels to binary (0/1)
clean_df["is_toxic"] = (clean_df["is_toxic"] > 0).astype(int)

# Remove duplicates
clean_df = clean_df.drop_duplicates(subset="text")
clean_df = clean_df.reset_index(drop=True)
print("\nClean Shape:", clean_df.shape)

# Distribution
print("\nToxic vs Non-toxic distribution:")
print(clean_df["is_toxic"].value_counts())

# Random samples
print("\n--- Random Samples ---")
print(clean_df.sample(5).values)

# Hinglish detection preview
hinglish_like = clean_df[
    clean_df["text"].str.contains(
        r'[\u0900-\u097F]|tum|hai|yeh|kyu|nahi|mera|tera|bhai|kaun',
        case=False,
        na=False
    )
]

if len(hinglish_like) > 0:
    print("\n--- Hinglish / Indian-style Samples ---")
    print(hinglish_like.sample(min(5, len(hinglish_like)))["text"].values)
else:
    print("\n(No obvious Hinglish patterns found — dataset mostly English)")

# Save clean dataset
os.makedirs("data", exist_ok=True)
OUTPUT_PATH = "data/final_moderation_dataset.csv"
clean_df.to_csv(OUTPUT_PATH, index=False)
print("\n✅ Saved clean dataset to:", OUTPUT_PATH)

Original Columns: ['text', 'hate_label', 'source', 'profanity_score', 'language', 'dataset_version', 'combined_date', 'text_length', 'word_count']
Original Shape: (29550, 9)
Detected text column: text
Detected label column: hate_label

Clean Shape: (29539, 2)

Toxic vs Non-toxic distribution:
is_toxic
0    15818
1    13721
Name: count, dtype: int64

--- Random Samples ---
[['holy crap that ching chong font' 1]
 ['suitor does any other administrator want to come and delete this idea for the sake of some shallow excuse like huh huh come on guys show us all how intelligent you are atta boy good boys'
  0]
 ['well then do not undo other people s edits when you clearly do not know what you are talking about you made a right vagina of that one mate'
  0]
 ['Sir aap Haryana me ho rahe rape ton dekhe #haryana' 0]
 ['sir inhone dikha diya aap hamla karke inki veins main kiska khoon behta hai ye bus ek bujurg par apni takat dikha sakte h'
  1]]

--- Hinglish / Indian-style Samples ---
['मै हिन्द

In [ ]:
# Dataset: combined_hate_speech_dataset.csv
# Model: Multilingual XLM-RoBERTa

!pip -q install transformers torch sentencepiece tqdm

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
from tqdm.auto import tqdm
import os

# Load dataset
DATA_PATH = "/content/data/final_moderation_dataset.csv"
df = pd.read_csv(DATA_PATH)
print("Dataset loaded:", df.shape)
print("Columns:", df.columns.tolist())

# Load model
MODEL_NAME = "unitary/multilingual-toxic-xlm-roberta"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print("Model loaded on:", device)

# Batched inference
def batch_detect(texts, batch_size=16):
    scores = []
    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Processing"):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=256
        ).to(device)
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.sigmoid(logits).cpu().numpy()
        batch_scores = np.max(probs, axis=1)
        batch_preds = (batch_scores >= 0.5).astype(int)
        scores.extend(batch_scores)
        preds.extend(batch_preds)
    return scores, preds

text_column = "text" if "text" in df.columns else df.columns[0]
texts = df[text_column].astype(str).tolist()
print("\nRunning toxicity detection on full dataset...")
scores, preds = batch_detect(texts, batch_size=16)
final_df = df.copy()
final_df["toxicity_score"] = np.round(scores, 4)
final_df["is_toxic_pred"] = preds

# Save results
os.makedirs("data/phase2", exist_ok=True)
OUTPUT_PATH = "data/phase2/dataset_with_toxicity_scores.csv"
final_df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved Phase-2 output to:", OUTPUT_PATH)

# Evaluation (using hate_label)
if "is_toxic" in final_df.columns:
    accuracy = (final_df["is_toxic"] == final_df["is_toxic_pred"]).mean()
    print("Approximate agreement with dataset labels:", round(accuracy, 3))
else:
    print("Ground-truth label not found; skipping accuracy.")

Dataset loaded: (29539, 2)
Columns: ['text', 'is_toxic']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/211 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

XLMRobertaForSequenceClassification LOAD REPORT from: unitary/multilingual-toxic-xlm-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Model loaded on: cuda

Running toxicity detection on full dataset...


Processing:   0%|          | 0/1847 [00:00<?, ?it/s]


Saved Phase-2 output to: data/phase2/dataset_with_toxicity_scores.csv
Approximate agreement with dataset labels: 0.616


In [ ]:
import bitsandbytes as bnb
print(bnb.__version__)

0.49.2


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)

True
12.8


In [ ]:
# Phase 3: Polite Rewriting using Mistral-7B (FULLY FIXED)

# ─────────────────────────────────────────
# 0. Install Required Libraries
# ─────────────────────────────────────────
!pip install -q -U bitsandbytes accelerate transformers sentencepiece

# ─────────────────────────────────────────
# 1. Imports
# ─────────────────────────────────────────
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm
import os

# ─────────────────────────────────────────
# 2. Check GPU
# ─────────────────────────────────────────
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ─────────────────────────────────────────
# 3. Load Data
# ─────────────────────────────────────────
DATA_PATH = "data/phase2/dataset_with_toxicity_scores.csv"
df = pd.read_csv(DATA_PATH)

toxic_df = df[df["is_toxic_pred"] == 1].copy().reset_index(drop=True)

# Limit to 100 samples (safe for Colab GPU)
toxic_df = toxic_df.sample(
    min(100, len(toxic_df)),
    random_state=42
).reset_index(drop=True)

print(f"Toxic samples selected: {len(toxic_df)}")

# ─────────────────────────────────────────
# 4. Load Mistral Model (4-bit Quantized)
# ─────────────────────────────────────────
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

print("\nLoading Mistral model (2-5 mins)...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16
)

model.eval()

print("Model loaded successfully!")

# ─────────────────────────────────────────
# 5. Rewrite Function (Correct Chat Template)
# ─────────────────────────────────────────
def rewrite_text(text):

    messages = [
        {
            "role": "system",
            "content": (
                "Rewrite toxic text into polite respectful text. "
                "Keep same meaning and same language. "
                "Only output rewritten sentence."
            )
        },
        {
            "role": "user",
            "content": text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True
        )

    decoded = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    rewritten = decoded[len(prompt):].strip()

    return rewritten

# ─────────────────────────────────────────
# 6. Run Rewriting
# ─────────────────────────────────────────
rewritten_texts = []

print("\nRewriting toxic comments using Mistral...")

for text in tqdm(toxic_df["text"].tolist()):
    try:
        rewritten = rewrite_text(text)
    except Exception as e:
        rewritten = "[ERROR] " + text

    rewritten_texts.append(rewritten)

toxic_df["rewritten_text"] = rewritten_texts

# ─────────────────────────────────────────
# 7. Stats
# ─────────────────────────────────────────
failed = sum(
    1 for t in rewritten_texts
    if t.startswith("[ERROR]")
)

success = len(rewritten_texts) - failed

print("\nRewrite Stats:")
print("Success:", success)
print("Failed :", failed)

# ─────────────────────────────────────────
# 8. Save Output
# ─────────────────────────────────────────
os.makedirs("data/phase3", exist_ok=True)

OUTPUT_PATH = "data/phase3/sample_100_rewrites_mistral.csv"

toxic_df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved to:", OUTPUT_PATH)

# ─────────────────────────────────────────
# 9. Preview Results
# ─────────────────────────────────────────
print("\n===== SAMPLE OUTPUT =====")

for i in range(min(10, len(toxic_df))):
    print("\nORIGINAL :", toxic_df.loc[i, "text"])
    print("REWRITE  :", toxic_df.loc[i, "rewritten_text"])
    print("-" * 60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 34.3 MB/s eta 0:00:00
GPU Available: True
GPU: Tesla T4
Toxic samples selected: 100

Loading Mistral model (2-5 mins)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded successfully!

Rewriting toxic comments using Mistral...


  0%|          | 0/100 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o


Rewrite Stats:
Success: 100
Failed : 0

Saved to: data/phase3/sample_100_rewrites_mistral.csv

===== SAMPLE OUTPUT =====

ORIGINAL : did you see the vma s i never thought fat joe what a slut going to come out with something i never felt so protected security courtesy of g unit that what a slut funny right after performed he called fat joe a pussy boy and almost seem like he wanted to cry this is turning out to be some crazy sh t
REWRITE  : you witness Fat Joe's unexpected behavior at the VMAs? I must admit, I was taken aback when he made some rather strong comments towards Fat Joe following his performance. It seemed as though he was feeling quite vulnerable and emotional, even appearing close to tears. The security provided by G Unit certainly added an element of intrigue to the situation. Quite an unusual turn of events
------------------------------------------------------------

ORIGINAL : Domada maulana apne allah ko yaad kar le jab choti si 6 sal ki bacchi ka rape karne wale ko 

In [ ]:
import pandas as pd

DATA_PATH = "/content/data/phase3/sample_100_rewrites.csv"

toxic_df = pd.read_csv(DATA_PATH)

print("Loaded:", toxic_df.shape)
toxic_df.head()

Loaded: (100, 5)


,text,is_toxic,toxicity_score,is_toxic_pred,rewritten_text
0,did you see the vma s i never thought fat joe ...,1,0.9839,1,Please rephrase the sentence politely:\n- Keep...
1,Domada maulana apne allah ko yaad kar le jab c...,1,0.5468,1,"Please rewrite the sentence politely, keeping ..."
2,there is no one to flirt with mad bitches in m...,0,0.8716,1,"Please, rephrase the sentence politely.\n- Kee..."
3,bet me glad you did not possess the balls to l...,1,0.8928,1,We are pleased to inform you that we have succ...
4,right it seems the bully editors are being arr...,1,0.9846,1,Polite rewriting:\n\nWe appreciate your valuab...


In [ ]:
# =========================================================
# FINAL EVALUATION (ALL-IN-ONE CELL)
# =========================================================

# Install (only if needed)
!pip -q install transformers sentence-transformers tqdm

# ---------------------------------------------------------
# 1. Imports
# ---------------------------------------------------------
import pandas as pd
import torch
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util

# ---------------------------------------------------------
# 2. Load Phase-3 Output
# ---------------------------------------------------------
DATA_PATH = "/content/data/phase3/sample_100_rewrites.csv"
toxic_df = pd.read_csv(DATA_PATH)

print("Dataset Loaded:", toxic_df.shape)

# Safety check
assert "text" in toxic_df.columns
assert "rewritten_text" in toxic_df.columns

# ---------------------------------------------------------
# 3. Load Toxicity Model (Same as Phase-2)
# ---------------------------------------------------------
MODEL_NAME = "unitary/multilingual-toxic-xlm-roberta"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Toxicity model loaded on:", device)

# ---------------------------------------------------------
# 4. Toxicity Scoring Function
# ---------------------------------------------------------
def get_toxicity_scores(texts, batch_size=16):
    scores = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Toxicity"):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=256
        ).to(device)

        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.sigmoid(logits).cpu().numpy()

        scores.extend(np.max(probs, axis=1))

    return scores

# ---------------------------------------------------------
# 5. Compute Toxicity Reduction
# ---------------------------------------------------------
orig_scores = get_toxicity_scores(toxic_df["text"].tolist())
rewr_scores = get_toxicity_scores(toxic_df["rewritten_text"].tolist())

toxic_df["orig_toxicity"] = orig_scores
toxic_df["rewr_toxicity"] = rewr_scores

avg_before = np.mean(orig_scores)
avg_after = np.mean(rewr_scores)
reduction = ((avg_before - avg_after) / avg_before) * 100

# ---------------------------------------------------------
# 6. Semantic Similarity
# ---------------------------------------------------------
sim_model = SentenceTransformer('all-MiniLM-L6-v2')

similarities = []

for i in tqdm(range(len(toxic_df)), desc="Similarity"):
    emb1 = sim_model.encode(toxic_df.loc[i, "text"], convert_to_tensor=True)
    emb2 = sim_model.encode(toxic_df.loc[i, "rewritten_text"], convert_to_tensor=True)
    sim = util.cos_sim(emb1, emb2).item()
    similarities.append(sim)

toxic_df["similarity"] = similarities

# ---------------------------------------------------------
# 7. Length Ratio
# ---------------------------------------------------------
toxic_df["len_ratio"] = (
    toxic_df["rewritten_text"].str.len() /
    toxic_df["text"].str.len()
)

# ---------------------------------------------------------
# 8. Final Results
# ---------------------------------------------------------
print("\n========== FINAL METRICS ==========")
print(f"Toxicity Before : {avg_before:.4f}")
print(f"Toxicity After  : {avg_after:.4f}")
print(f"Reduction (%)   : {reduction:.2f}%")
print(f"Similarity      : {np.mean(similarities):.4f}")
print(f"Length Ratio    : {toxic_df['len_ratio'].mean():.4f}")
print(f"Success Rate    : 100%")

# ---------------------------------------------------------
# 9. Save Evaluated File
# ---------------------------------------------------------
OUTPUT_PATH = "/content/data/phase3/final_evaluated_results.csv"
toxic_df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved Final Results to:", OUTPUT_PATH)

Dataset Loaded: (100, 5)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: unitary/multilingual-toxic-xlm-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Toxicity model loaded on: cuda


Toxicity:   0%|          | 0/7 [00:00<?, ?it/s]

Toxicity:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similarity:   0%|          | 0/100 [00:00<?, ?it/s]


========== FINAL METRICS ==========
Toxicity Before : 0.8199
Toxicity After  : 0.1538
Reduction (%)   : 81.24%
Similarity      : 0.4106
Length Ratio    : 2.1721
Success Rate    : 100%

Saved Final Results to: /content/data/phase3/final_evaluated_results.csv
